Instalamos dependencias

In [1]:
!pip install sweetviz

In [2]:
!pip install ydata-profiling

In [3]:
from wget import download
from os import path, listdir
import pandas as pd

Descargamos la base

In [4]:
if not path.exists("schema.csv"):
  download("https://media.githubusercontent.com/media/StackExchange/Survey/refs/heads/main/packages/archive/2025/schema.csv")
if not path.exists("results.csv"):
  download("https://media.githubusercontent.com/media/StackExchange/Survey/6ae2209cd35a55e92739ba82f7887fb86807a069/packages/archive/2025/results.csv")

In [12]:
# Filtramos las columnas
import re
raw_dataset_results = pd.read_csv("results.csv",low_memory=False)
schema_dataset = pd.read_csv("schema.csv")

# 1. Definición de las listas de columnas para cada subgrupo
cols_g1 = ['Country', 'WorkExp', 'ConvertedCompYearly', 'CompTotal', 'Currency', 'Age', 'EdLevel', 'Employment', 'YearsCode']
cols_g2 = ['WorkExp', 'AIAcc', 'JobSat', 'AISent', 'AISelect', 'AIThreat', 'AILearnHow']

# Para el Grupo 3 extraemos dinámicamente todas las de JobSatPoints (del 1 al 15 + TEXT) para evitar hardcodearlas
cols_jobsat = [col for col in raw_dataset_results.columns if col.startswith('JobSatPoints')]
cols_g3 = ['NewRole', 'JobSat'] + cols_jobsat

cols_g4 = ['ConvertedCompYearly', 'WorkExp', 'OrgSize', 'AISelect', 'RemoteWork', 'DevType', 'Industry', 'ICorPM']

# 2. Creación de los DataFrames independientes para cada subgrupo
# Usamos .copy() para que cada compañero pueda limpiar nulos o transformar datos sin afectar el dataset original

# Subgrupo 1: Perfil Económico (Para Hipótesis A y HU1)
df_perfil_economico = raw_dataset_results[cols_g1].copy()

# Subgrupo 2: Impacto de la IA (Para Hipótesis B y HB1)
df_impacto_ia = raw_dataset_results[cols_g2].copy()

# Subgrupo 3: Cambio de Rol (Para Hipótesis C)
df_cambio_rol = raw_dataset_results[cols_g3].copy()

# Subgrupo 4: Premio Salarial (Para Hipótesis M4)
df_premio_salarial = raw_dataset_results[cols_g4].copy()

Abrimos los archivos

Imprimimos el resumen estadistico

In [13]:
raw_dataset_results.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49191 entries, 0 to 49190
Columns: 172 entries, ResponseId to JobSat
dtypes: float64(52), int64(1), object(119)
memory usage: 64.6+ MB


Vemos solamente las de tipo float

In [8]:
raw_dataset_results.select_dtypes(include="float64").columns

Index(['WorkExp', 'YearsCode', 'TechEndorse_1', 'TechEndorse_2',
       'TechEndorse_3', 'TechEndorse_4', 'TechEndorse_5', 'TechEndorse_6',
       'TechEndorse_7', 'TechEndorse_8', 'TechEndorse_9', 'TechEndorse_13',
       'TechOppose_1', 'TechOppose_2', 'TechOppose_3', 'TechOppose_5',
       'TechOppose_7', 'TechOppose_9', 'TechOppose_11', 'TechOppose_13',
       'TechOppose_16', 'TechOppose_15', 'JobSatPoints_1', 'JobSatPoints_2',
       'JobSatPoints_3', 'JobSatPoints_4', 'JobSatPoints_5', 'JobSatPoints_6',
       'JobSatPoints_7', 'JobSatPoints_8', 'JobSatPoints_9', 'JobSatPoints_10',
       'JobSatPoints_11', 'JobSatPoints_13', 'JobSatPoints_14',
       'JobSatPoints_15', 'JobSatPoints_16', 'ToolCountWork',
       'ToolCountPersonal', 'CompTotal', 'SO_Actions_1', 'SO_Actions_16',
       'SO_Actions_3', 'SO_Actions_4', 'SO_Actions_5', 'SO_Actions_6',
       'SO_Actions_9', 'SO_Actions_7', 'SO_Actions_10', 'SO_Actions_15',
       'ConvertedCompYearly', 'JobSat'],
      dtype='object

Vemos solamente las de tipo string

In [14]:
raw_dataset_results.select_dtypes(include="object").columns

Index(['MainBranch', 'Age', 'EdLevel', 'Employment', 'EmploymentAddl',
       'LearnCodeChoose', 'LearnCode', 'LearnCodeAI', 'AILearnHow', 'DevType',
       ...
       'AIAgentKnowledge', 'AIAgentKnowWrite', 'AIAgentOrchestration',
       'AIAgentOrchWrite', 'AIAgentObserveSecure', 'AIAgentObsWrite',
       'AIAgentExternal', 'AIAgentExtWrite', 'AIHuman', 'AIOpen'],
      dtype='object', length=119)

Vemos las filas columnas que no estan representadas en schema.csv

In [18]:
float_cols_dataset = set(raw_dataset_results.select_dtypes(include="float64").columns)
string_cols_dataset = set(raw_dataset_results.select_dtypes(include="object").columns)
int_cols_dataset = set(raw_dataset_results.select_dtypes(include="int64").columns)

cols_dataset = set(raw_dataset_results.columns)
cols_schema  = set(schema_dataset["qname"])

float_faltan_en_schema =  float_cols_dataset - cols_schema
string_faltan_en_schema = string_cols_dataset - cols_schema
int_faltan_en_schema = int_cols_dataset - cols_schema

print(f"Faltan en el esquema: {len(float_faltan_en_schema)} columnas float")
for falta in sorted(float_faltan_en_schema):
    print(falta)
print("-" * 90)

print(f"Faltan en el esquema: {len(int_faltan_en_schema)} columnas int")
for falta in sorted(int_faltan_en_schema):
    print(falta)
print("-" * 90)

print(f"Faltan en el esquema: {len(string_faltan_en_schema)} columnas string")
for falta in sorted(string_faltan_en_schema):
    print(falta)
print("-" * 90)


faltan_en_schema = cols_dataset - cols_schema
print(f"Faltan en el esquema: {len(faltan_en_schema)} columnas")
for falta in sorted(faltan_en_schema):
    print(falta)
print("-" * 90)
no_existen_en_dataset = cols_schema - cols_dataset
print(f"No existen en el dataset: {len(no_existen_en_dataset)}")
for no_existe in sorted(no_existen_en_dataset):
    print(no_existe)

Faltan en el esquema: 1 columnas float
ConvertedCompYearly
------------------------------------------------------------------------------------------
Faltan en el esquema: 1 columnas int
ResponseId
------------------------------------------------------------------------------------------
Faltan en el esquema: 44 columnas string
AIAgentChallengesNeutral
AIAgentChallengesSomewhat agree
AIAgentChallengesSomewhat disagree
AIAgentChallengesStrongly agree
AIAgentChallengesStrongly disagree
AIAgentImpactNeutral
AIAgentImpactSomewhat agree
AIAgentImpactSomewhat disagree
AIAgentImpactStrongly agree
AIAgentImpactStrongly disagree
AIModelsAdmired
AIModelsHaveWorkedWith
AIModelsWantToWorkWith
AIToolCurrently mostly AI
AIToolCurrently partially AI
AIToolDon't plan to use AI for this task
AIToolPlan to mostly use AI
AIToolPlan to partially use AI
CommPlatformAdmired
CommPlatformHaveWorkedWith
CommPlatformWantToWorkWith
DatabaseAdmired
DatabaseHaveWorkedWith
DatabaseWantToWorkWith
DevEnvsAdmired
DevE

In [23]:
df_perfil_economico.isna().sum()

Country                13754
WorkExp                 6298
ConvertedCompYearly    25244
CompTotal              24325
Currency               13754
Age                        0
EdLevel                 1042
Employment               852
YearsCode               6149
dtype: int64

In [25]:
df_impacto_ia.isna().sum()

WorkExp        6298
AIAcc         15894
JobSat        22521
AISent        15724
AISelect      15471
AIThreat      13113
AILearnHow    20934
dtype: int64

In [26]:
df_cambio_rol.isna().sum()

NewRole                 13664
JobSat                  22521
JobSatPoints_1          24248
JobSatPoints_2          24248
JobSatPoints_3          24248
JobSatPoints_4          24248
JobSatPoints_5          24248
JobSatPoints_6          24248
JobSatPoints_7          24248
JobSatPoints_8          24248
JobSatPoints_9          24248
JobSatPoints_10         24248
JobSatPoints_11         24248
JobSatPoints_13         24248
JobSatPoints_14         24248
JobSatPoints_15         24248
JobSatPoints_16         24248
JobSatPoints_15_TEXT    48527
dtype: int64

In [24]:
df_premio_salarial.isna().sum()

ConvertedCompYearly    25244
WorkExp                 6298
OrgSize                15013
AISelect               15471
RemoteWork             15411
DevType                 5511
Industry               15549
ICorPM                 15948
dtype: int64